In [0]:
from pyspark.sql.functions import *

#try:
# Configuration for accessing CosmosDB
readConfig = {
    "spark.cosmos.accountEndpoint": "https://scc58569.documents.azure.com:443/",
    "spark.cosmos.accountKey": "Db0g6Zbdb7P9MTPhMppUn4toDTloc9a5p0323SavkQ2qM9HWSeipOJzRHLjo3BiQByHtN99tGDxKzVc0PLc4Fw==",
    "spark.cosmos.database": "scc23db",
    "spark.cosmos.container": "auctions",
    # Getting only the id and owner from the auctions
    "spark.cosmos.read.customQuery": "SELECT a.id, a.ownerId, a.status, a.bids FROM auctions a"
}

auctions = spark.read.format("cosmos.oltp").options(**readConfig) \
                    .option("spark.cosmos.read.inferSchema.enabled", "true") \
                    .load()
# Let's register the dataframe as a view
auctions.createOrReplaceTempView("auctions")

# Write configuration
writeConfig = {
    "spark.cosmos.accountEndpoint": "https://scc58569.documents.azure.com:443/",
    "spark.cosmos.accountKey": "Db0g6Zbdb7P9MTPhMppUn4toDTloc9a5p0323SavkQ2qM9HWSeipOJzRHLjo3BiQByHtN99tGDxKzVc0PLc4Fw==",
    "spark.cosmos.database": "scc23db",
    "spark.cosmos.container": "auctionsTrend",
}

table_values = []

result = spark.sql("""SELECT id, bids, status FROM auctions where status = "OPEN" """)
table_values = result.collect()

d: dict = {}
# for each row, count the number of non null bids
for row in table_values:
    n_bids = 0
    for bid in row.bids:
        if bid:
            n_bids += 1
    if n_bids == 0:
        continue
    d[row.id]= n_bids

# sort by number of bids, higher values come first
d = sorted(d.items(), key=lambda item: item[1], reverse=True)

df = spark.createDataFrame(data=d, schema = ["id","bid_count"])
df.show()

#Write to Cosmos DB from the result DataFrame
df.write.format("cosmos.oltp").options(**writeConfig).mode("append").save()
#except Exception as e:
#    print(e)

+--------------------+---------+
|                  id|bid_count|
+--------------------+---------+
|916552e7-8918-479...|        3|
|0172accf-9e26-46c...|        3|
|c9f2863f-d7c1-4c6...|        3|
|c75b457e-57db-482...|        3|
|2c639375-117a-449...|        3|
|175b663a-bc19-4a9...|        3|
|0bd81da8-509e-4d7...|        3|
|b5ab47c7-2e2f-4b8...|        3|
|131d81f4-11de-49c...|        3|
|784d7d78-aa00-4e5...|        3|
|f17721f7-a5b3-47b...|        3|
|e6a673d9-4ae0-476...|        3|
|5232ca39-f7a8-436...|        3|
|7cab20e2-5a0f-468...|        3|
|5ebf7dd9-cf12-4f6...|        2|
|2e77d2b5-bcac-459...|        2|
|2c3138ef-423f-42f...|        2|
|72a03c76-22e0-478...|        2|
|4fc589ee-67e5-4a3...|        2|
|0e3006e4-ff03-476...|        2|
+--------------------+---------+
only showing top 20 rows

